# Labor Rights RAG Assistant

## Project Goal:

A Retrieval-Augmented Generation pipeline for answering employment law questions in Arabic and English, grounded strictly in a curated corpus of Lebanese labor law guides. The system combines hybrid retrieval, cross-encoder reranking, and language-aware generation with a full bilingual evaluation suite.

## Pipeline Overview

1. **Document Loading & Chunking** — PDF guides are loaded, blank pages removed, and language-tagged by filename. Text is split into token-aware overlapping chunks (500 tokens, 100 overlap) using a recursive splitter.

2. **Embedding Generation** — Chunks are encoded with `intfloat/multilingual-e5-base` using E5's required `passage:` prefix. Embeddings are L2-normalized for cosine similarity via inner product.

3. **Hybrid Retrieval (FAISS + BM25)** — Dense vectors are indexed with `IndexFlatIP`. A BM25 index provides keyword matching with language-aware tokenization. Scores are combined with language-specific weights (EN: 0.7/0.3, AR: 0.5/0.5).

4. **Language-Aware Filtering** — Query language is detected via Unicode range analysis. Retrieval is strictly filtered to same-language chunks to prevent cross-language mixing.

5. **Cross-Encoder Reranking** — Retrieved candidates are reranked with `mmarco-mMiniLMv2-L12-H384-v1`, scoring each query-chunk pair jointly. Final context: top 3 chunks (English), top 5 (Arabic).

6. **Grounded Generation** — Top chunks are injected into a language-specific prompt. `gpt-4.1-mini` generates answers strictly from context with an explicit refusal mechanism when the answer is not found.

## Design Choices

- **Hybrid retrieval** combines semantic search (dense) with keyword matching (sparse), improving robustness on short or terminology-heavy legal queries where either alone underperforms.

- **Cross-encoder reranking** re-scores retrieved candidates with full query-document attention, producing a higher-quality final context than embedding similarity alone.

- **Token-aware chunking** uses tiktoken to enforce consistent chunk sizes across Arabic and English, avoiding the fragmentation that character-based splitting causes on Arabic text.

- **Refusal mechanism** constrains the LLM to answer only from retrieved context, eliminating hallucination on out-of-scope queries — validated at 100% refusal accuracy on both languages.

- **Language-specific retrieval parameters** (hybrid weights, final top-K) account for the performance gap between Arabic and English retrieval, rather than applying a single configuration to both.

## Imports

In [1]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from openai import OpenAI

import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

## Setup

In [2]:
load_dotenv()

# Set API key 
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found in .env file")

# Define file path
DATA_DIR = "data"

if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(f"{DATA_DIR} directory not found")

print("Environment loaded successfully.")

Environment loaded successfully.


## Configuration

In [28]:
CONFIG = {
    "top_k_retrieval": 10,
    "top_k_final": 3,
    "candidate_multiplier": 5,
    "use_reranker": True,
    "dense_weight":    {"en": 0.7, "ar": 0.5},
    "sparse_weight":   {"en": 0.3, "ar": 0.5},
}

## Model Initialization

In [4]:
os.environ["HF_HUB_OFFLINE"] = "1"

# Embedding model: multilingual E5
EMBED_MODEL_NAME = "intfloat/multilingual-e5-base"
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

# Reranker model: multilingual cross-encoder
RERANK_MODEL_NAME = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
reranker = CrossEncoder(RERANK_MODEL_NAME)

print(f"Embedding model loaded: {EMBED_MODEL_NAME}")
print(f"Reranker model loaded:  {RERANK_MODEL_NAME}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded: intfloat/multilingual-e5-base
Reranker model loaded:  cross-encoder/mmarco-mMiniLMv2-L12-H384-v1


## PDF Loading

In [5]:
loader = PyPDFDirectoryLoader(DATA_DIR)
documents = loader.load()

if not documents:
    raise ValueError("No documents were loaded. Check your data folder.")

print(f"Total pages loaded (including blanks): {len(documents)}")

# Remove empty pages
documents = [doc for doc in documents if doc.page_content.strip()]

print(f"Pages after removing blanks: {len(documents)}")

if not documents:
    raise ValueError("All pages are empty after cleaning.")

# Tag each document with its language based on filename
def detect_language_from_filename(filepath):
    filename = os.path.basename(filepath)
    # If filename contains Arabic characters, it's Arabic
    return "ar" if any('\u0600' <= c <= '\u06FF' for c in filename) else "en"

for doc in documents:
    lang = detect_language_from_filename(doc.metadata.get("source", ""))
    doc.metadata["language"] = lang

# Show language distribution
from collections import Counter
lang_counts = Counter(doc.metadata["language"] for doc in documents)
print(f"Language distribution: {dict(lang_counts)}")

# Show sample from first valid page
sample = documents[44]

print("\nSample Preview:")
print(sample.page_content[:500])

print("\nMetadata:", sample.metadata)

Total pages loaded (including blanks): 192
Pages after removing blanks: 189
Language distribution: {'en': 95, 'ar': 94}

Sample Preview:
CREDITS 
Acknowledgements:
Legal guide authored by Layal Abou Daher, legal consultant. Edited by Martin Clutterbuck/NRC. January 2024.
Design:
 
All photos @NRC and other external photographers as credited. 
Cover Photo: Palestinian Baker, (Photo: Racha El Daoi/NRC)
Back Cover Photo: Making Falafel, Bekaa (Photo: Racha El Dao/NRC)

Metadata: {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.0 (Windows)', 'creationdate': '2024-01-09T12:50:11+03:00', 'moddate': '2024-01-09T13:37:41+03:00', 'trapped': '/False', 'source': 'data\\Self-Employment-Guide.pdf', 'total_pages': 52, 'page': 1, 'page_label': '2', 'language': 'en'}


In [6]:
# Show unique sources and their detected language

seen = {}
for doc in documents:
    src = os.path.basename(doc.metadata["source"])
    lang = doc.metadata["language"]
    seen[src] = lang

for filename, lang in seen.items():
    print(f"{filename}  -  language: {lang}")

Employment-Rights-Guide.pdf  -  language: en
Self-Employment-Guide.pdf  -  language: en
دليل العمل الحر والمشاريع الصغيرة في لبنان .pdf  -  language: ar
دليل عن حقوق العمل في لبنان.pdf  -  language: ar


## Chunking

In [7]:
import tiktoken

# Token counter
encoding = tiktoken.get_encoding("cl100k_base")

def token_length(text):
    return len(encoding.encode(text))

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # tokens 
    chunk_overlap=100,
    length_function=token_length,
    separators=["\n\n", "\n", ".", " ", "", "? ", "! ","،"]
)

chunks = splitter.split_documents(documents)

# Add chunk ID
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i
    
print(f"Total chunks created: {len(chunks)}")

# Verify language tags survived the split
lang_counts = Counter(chunk.metadata.get("language", "unknown") for chunk in chunks)
print(f"Chunks by language: {dict(lang_counts)}")

print(f"\nSample English chunk:")
en_sample = next(c for c in chunks if c.metadata.get("language") == "en")
print(en_sample.page_content[:300])
print("Metadata:", en_sample.metadata)

print(f"\nSample Arabic chunk:")
ar_sample = next(c for c in chunks if c.metadata.get("language") == "ar")
print(ar_sample.page_content[:300])
print("Metadata:", ar_sample.metadata)

Total chunks created: 520
Chunks by language: {'en': 168, 'ar': 352}

Sample English chunk:
Acknowledgements. Legal guide authored by NRC and El Meouchi law firm ( https://www.elmeouchi.
com/  Chadia El Meouchi and Elie El Feghali), facilitated by TrustLaw, the Thomson Reuters Foundation’s 
global pro bono service. Edited by Martin Clutterbuck/NRC. Special thanks to Lianna Badamo, Sarah 
G
Metadata: {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.1 (Windows)', 'creationdate': '2023-03-16T15:35:55+03:00', 'moddate': '2023-03-18T09:26:14+03:00', 'trapped': '/False', 'source': 'data\\Employment-Rights-Guide.pdf', 'total_pages': 44, 'page': 1, 'page_label': '2', 'language': 'en', 'chunk_id': 0}

Sample Arabic chunk:
في لبنان
دليل العمل الحر 
والمشاريع الصغيرة
Metadata: {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.0 (Windows)', 'creationdate': '2024-03-19T13:22:30+03:00', 'moddate': '2024-04-12T14:07:51+02:00', 'title': 'Self-Employment and Small

## Hybrid Search

In [8]:
from rank_bm25 import BM25Okapi
import re

def tokenize_for_bm25(text, lang="en"):
    if lang == "ar":
        # Strip punctuation, normalize, split on whitespace
        text = re.sub(r'[^\w\s\u0600-\u06FF]', ' ', text)
        return text.strip().split()
    else:
        return text.lower().split()

tokenized_corpus = [
    tokenize_for_bm25(chunk.page_content, lang=chunk.metadata.get("language", "en"))
    for chunk in chunks
]

bm25 = BM25Okapi(tokenized_corpus)
print("BM25 index built with language-aware tokenization.")

BM25 index built with language-aware tokenization.


## Generate Embeddings

In [9]:
# E5 models require a prefix: 'passage' for documents, 'query' for queries.
def get_embeddings(texts, prefix="passage"):
    prefixed = [f"{prefix}: {t}" for t in texts]
    embeddings = embed_model.encode(
        prefixed,
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True   
    )
    return np.array(embeddings).astype("float32")

# Extract text from chunks
texts = [chunk.page_content for chunk in chunks]

# Generate corpus embeddings
embeddings = get_embeddings(texts, prefix="passage")

embeddings = np.array(embeddings).astype("float32")

dimension = embeddings.shape[1]
print(f"Embedding dimension: {dimension}")
print(f"Total embeddings: {embeddings.shape[0]}")

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Embedding dimension: 768
Total embeddings: 520


## Create FAISS Index

In [10]:
import pickle

if os.path.exists("index.faiss") and os.path.exists("chunks.pkl"):
    # Load existing index
    index = faiss.read_index("index.faiss")
    with open("chunks.pkl", "rb") as f:
        chunks = pickle.load(f)
    print(f"Loaded existing FAISS index: {index.ntotal} vectors")
    print(f"Loaded {len(chunks)} chunks from disk")

else:
    # Build index from scratch
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)
    print(f"FAISS index built with {index.ntotal} vectors")

FAISS index built with 520 vectors


## Save FAISS Index

In [11]:
# Save FAISS index (both languages)
faiss.write_index(index, "index.faiss")

# Save all chunks
with open("chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

# Save language-separated chunks 
en_chunks = [c for c in chunks if c.metadata.get("language") == "en"]
ar_chunks = [c for c in chunks if c.metadata.get("language") == "ar"]

with open("chunks_en.pkl", "wb") as f:
    pickle.dump(en_chunks, f)

with open("chunks_ar.pkl", "wb") as f:
    pickle.dump(ar_chunks, f)

print(f"Saved {len(chunks)} total chunks")
print(f"  English chunks: {len(en_chunks)}")
print(f"  Arabic chunks:  {len(ar_chunks)}")

Saved 520 total chunks
  English chunks: 168
  Arabic chunks:  352


## Functions

In [29]:
# Language detection function

def detect_language(text):
    arabic_chars = sum(1 for c in text if '\u0600' <= c <= '\u06FF')
    latin_chars = sum(1 for c in text if c.isascii() and c.isalpha())
    if arabic_chars > latin_chars:
        return "ar"
    else:
        return "en"


# Retrieval function

def retrieve(query, k=5, candidate_multiplier=5, lang_filter=True):
    query_lang = detect_language(query)

    # Dense retrieval (FAISS) to retrive top candidates from FAISS
    query_embedding = get_embeddings([query], prefix="query")
    dense_scores, dense_ids = index.search(query_embedding, k * candidate_multiplier)

    # Build O(1) lookup dict instead of scanning the list each iteration
    dense_map = {
        int(idx): float(score)
        for idx, score in zip(dense_ids[0], dense_scores[0])
        if idx != -1  # FAISS returns -1 for empty slots
    }

    # Sparse retrieval (BM25) 
    tokenized_query = tokenize_for_bm25(query, lang=query_lang)
    sparse_scores = bm25.get_scores(tokenized_query)

    results = []

    for i, chunk in enumerate(chunks):
        chunk_lang = chunk.metadata.get("language", "en")

        # Language filter (skip chunks that don't match the query language)
        if lang_filter and chunk_lang != query_lang:
            continue

        dense_score = dense_map.get(i, 0.0)
        sparse_score = float(sparse_scores[i])

        combined_score = (
            CONFIG["dense_weight"][query_lang] * dense_score +
            CONFIG["sparse_weight"][query_lang] * sparse_score
        )

        results.append({
            "score": combined_score,
            "source": os.path.basename(chunk.metadata.get("source", "")),
            "page": chunk.metadata.get("page_label"),
            "content": chunk.page_content,
            "language": chunk_lang
        })

    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:k * 2]  # wide pool for reranker


# Rerank candidates using a cross-encoder.
# Cross-encoder scores each (query, passage) pair jointly

def rerank(query, candidates, top_n=3):
    if not candidates:
        return []

    pairs = [[query, doc["content"]] for doc in candidates]
    scores = reranker.predict(pairs)

    for doc, score in zip(candidates, scores):
        doc["rerank_score"] = float(score)

    reranked = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)
    return reranked[:top_n]

In [63]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


def build_prompt(query, retrieved_docs, query_lang):
    """Build the system message and user prompt for a given language."""
    context = "\n\n".join(
        f"[Source {i+1}]\n{doc['content']}"
        for i, doc in enumerate(retrieved_docs)
    )

    if query_lang == "ar":
        system_msg = "أنت مساعد قانوني دقيق. أجب فقط باللغة العربية ولا تستخدم أي لغة أخرى."
        refusal_phrase = "الدليل لا يوفر هذه المعلومات"
        user_prompt = f"""أنت مساعد قانوني تجيب على الأسئلة بناءً على السياق فقط.

القواعد:
- استخدم فقط المعلومات الموجودة في السياق.
- إذا لم تجد الإجابة، قل: "{refusal_phrase}"
- لا تخترع معلومات.

السياق:
{context}

السؤال:
{query}

الجواب:"""
    else:
        system_msg = "You are a precise legal assistant. Answer ONLY in English."
        refusal_phrase = "The guide does not provide this information"
        user_prompt = f"""You are a legal assistant answering strictly from context.

Rules:
- Use only the provided context.
- If missing, say: "{refusal_phrase}"
- Do not hallucinate.

Context:
{context}

Question:
{query}

Answer:"""

    return system_msg, user_prompt, refusal_phrase


def call_llm(system_msg, user_prompt):
    """Send prompt to OpenAI and return the answer string."""
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0,
        max_tokens=500
    )
    return response.choices[0].message.content.strip()


def generate_answer(query):
    # Full RAG pipeline: retrieve → rerank → build prompt → generate.
    query_lang = detect_language(query)

    candidates = retrieve(
        query,
        k=CONFIG["top_k_retrieval"],
        candidate_multiplier=CONFIG["candidate_multiplier"]
    )

    top_n = 5 if query_lang == "ar" else CONFIG["top_k_final"]

    if CONFIG["use_reranker"]:
        retrieved_docs = rerank(query, candidates, top_n=top_n)
    else:
        retrieved_docs = candidates[:top_n]

    if not retrieved_docs:
        msg = "لم يتم العثور على وثائق ذات صلة." if query_lang == "ar" else "No relevant documents were retrieved."
        return {"answer": msg, "sources": []}

    system_msg, user_prompt, refusal_phrase = build_prompt(query, retrieved_docs, query_lang)
    answer = call_llm(system_msg, user_prompt)

    if refusal_phrase in answer:
        return {"answer": answer, "sources": []}

    seen = set()
    sources = []
    for doc in retrieved_docs:
        key = f"{doc['source']} (Page {doc['page']})"
        if key not in seen:
            seen.add(key)
            sources.append(key)

    return {"answer": answer, "sources": sources}

## Test RAG Pipeline on Sample Questions

In [64]:
evaluation_questions_en = [

    # EMPLOYMENT RIGHTS GUIDE (17)
    "What is the legally mandated number of annual leave days?",
    "How does annual leave increase based on years of service?",
    "What is the official minimum wage?",
    "What are the maximum legal working hours per week?",
    "How is overtime compensated?",
    "What are the rules governing sick leave?",
    "Is maternity leave paid and for how long?",
    "What rights exist during the probation period?",
    "What prior notice is required before termination?",
    "What compensation is due in case of abusive dismissal?",
    "Under what conditions can employment be terminated due to force majeure?",
    "What are the employer's obligations regarding social security contributions?",
    "Are foreign workers entitled to social security benefits?",
    "Are employees protected from dismissal while on leave?",
    "What are the procedures for challenging abusive dismissal?",
    "Does the guide regulate cryptocurrency salary payments?",
    "Does the guide contain rules about remote work or work-from-home policies?",
    
    # SELF-EMPLOYMENT & SMALL BUSINESS GUIDE (13)
    "How does the guide define workers not linked to an employer?",
    "Are self-employed individuals covered by social security?",
    "Are foreigners eligible for social security benefits?",
    "When is a daily worker considered an employee versus self-employed?",
    "Must individuals conducting commercial activities register as traders?",
    "What activities are considered commercial in nature?",
    "Are workers who work for multiple employers covered by social security?",
    "Are self-employed individuals required to pay tax?",
    "Are self-employed individuals required to pay municipal fees, and on what basis are they calculated?",
    "Are small-scale daily labourers considered traders under commercial law?",
    "Does the guide set minimum pricing standards for freelance services?",
    "Does the guide provide government grants or startup funding schemes for self-employed workers?",
    "What are the grounds under which an employer can dismiss an employee without indemnity or prior notice?"
]

In [65]:
# Arabic evaluation questions (same questions, same order)
evaluation_questions_ar = [

    # دليل حقوق العمل
    "ما هو العدد القانوني المقرر لأيام الإجازة السنوية؟",
    "كيف تزداد الإجازة السنوية بناءً على سنوات الخدمة؟",
    "ما هو الحد الأدنى الرسمي للأجور؟",
    "ما هو الحد الأقصى القانوني لساعات العمل في الأسبوع؟",
    "كيف يتم تعويض العمل الإضافي؟",
    "ما هي القواعد المتعلقة بالإجازة المرضية؟",
    "هل إجازة الأمومة مدفوعة الأجر وما مدتها؟",
    "ما هي الحقوق خلال فترة الاختبار؟",
    "ما هي مدة الإنذار المسبق المطلوبة قبل إنهاء العمل؟",
    "ما هو التعويض المستحق في حالة الفصل التعسفي؟",
    "في أي ظروف يمكن إنهاء العمل بسبب القوة القاهرة؟",
    "ما هي التزامات صاحب العمل تجاه اشتراكات الضمان الاجتماعي؟",
    "هل يحق للعمال الأجانب الاستفادة من الضمان الاجتماعي؟",
    "هل يُحمى الموظفون من الفصل أثناء الإجازة؟",
    "ما هي إجراءات الطعن في الفصل التعسفي؟",
    "هل ينظم الدليل دفع الرواتب بالعملات المشفرة؟",
    "هل يتضمن الدليل قواعد تتعلق بالعمل عن بُعد؟",

    # دليل العمل الحر والأعمال الصغيرة
    "كيف يعرّف الدليل العمال غير المرتبطين بصاحب عمل؟",
    "هل يشمل الضمان الاجتماعي العمال المستقلين؟",
    "هل يحق للأجانب الاستفادة من الضمان الاجتماعي؟",
    "متى يُعتبر العامل اليومي موظفاً بدلاً من عامل مستقل؟",
    "هل يجب على الأفراد الذين يمارسون النشاط التجاري التسجيل كتجار؟",
    "ما هي الأنشطة التي تُعدّ تجارية بطبيعتها؟",
    "هل يشمل الضمان الاجتماعي العمال الذين يعملون لدى أكثر من صاحب عمل؟",
    "هل يخضع العمال المستقلون لضريبة الدخل؟",
    "هل يخضع العمال المستقلون لرسوم بلدية وما هو أساس احتسابها؟",
    "هل يُعتبر العمال اليوميون صغار الحجم تجاراً وفق قانون التجارة؟",
    "هل يحدد الدليل معايير تسعير دنيا للخدمات المستقلة؟",
    "هل يوفر الدليل منحاً حكومية أو برامج تمويل للعمال المستقلين؟",
    "ما هي القطاعات التي يُسمح للعمال السوريين بالعمل فيها قانونياً في لبنان؟"
]

In [66]:
# Combined for running both
evaluation_questions = evaluation_questions_en + evaluation_questions_ar

print(f"English questions: {len(evaluation_questions_en)}")
print(f"Arabic questions:  {len(evaluation_questions_ar)}")
print(f"Total:             {len(evaluation_questions)}")

English questions: 30
Arabic questions:  30
Total:             60


In [67]:
import pandas as pd

def run_evaluation(questions, label):
    results = []

    for i, question in enumerate(questions):
        print(f"\n[{label.upper()} Q{i+1}] {question}")

        output = generate_answer(question)

        print(f"Answer: {output['answer'][:150]}...")

        results.append({
            "language": label,
            "question": question,
            "answer": output["answer"],
            "sources": ", ".join(output["sources"])
        })
        
    return results

In [68]:
results_en = run_evaluation(evaluation_questions_en, "en")
results_ar = run_evaluation(evaluation_questions_ar, "ar")
results_all = results_en + results_ar

print(f"Answers generated: {len(results_all)}")


[EN Q1] What is the legally mandated number of annual leave days?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Answer: The legally mandated number of annual leave days in Lebanon is 15 days fully paid, provided that the employee has been employed by the employer for at...

[EN Q2] How does annual leave increase based on years of service?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Answer: Annual leave increases based on years of service as follows:

- 15 working days for employees who have worked between one to five years.
- 17 working ...

[EN Q3] What is the official minimum wage?


...

Answer: الدليل لا يوفر هذه المعلومات....

[AR Q29] هل يوفر الدليل منحاً حكومية أو برامج تمويل للعمال المستقلين؟


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Answer: الدليل لا يوفر هذه المعلومات...

[AR Q30] ما هي القطاعات التي يُسمح للعمال السوريين بالعمل فيها قانونياً في لبنان؟


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Answer: يُسمح للعمال السوريين بالعمل قانونياً في لبنان في قطاعات الزراعة والبناء والبيئة. كما يُسمح لهم بالعمل كحمالين أو بوابين بحرية دون تطبيق نظام الحصص ال...
Answers generated: 60


In [69]:
# For clean display

from IPython.display import display, Markdown

def display_results(results, title):
    display(Markdown(f"## {title}"))

    for r in results:
        display(Markdown(f"### Question\n{r['question']}"))
        display(Markdown(f"**Answer:**\n\n{r['answer']}"))
        if r["sources"]:
            display(Markdown(f"**Sources:** {r['sources']}"))
        else:
            display(Markdown(f"**Sources:** None"))
        display(Markdown("---"))

display_results(results_en, "English Results")
display_results(results_ar, "Arabic Results")        

## English Results

### Question
What is the legally mandated number of annual leave days?

**Answer:**

The legally mandated number of annual leave days in Lebanon is 15 days fully paid, provided that the employee has been employed by the employer for at least one year.

**Sources:** Employment-Rights-Guide.pdf (Page 16), Employment-Rights-Guide.pdf (Page 24), Employment-Rights-Guide.pdf (Page 17)

---

...

---

### Question
ما هي القطاعات التي يُسمح للعمال السوريين بالعمل فيها قانونياً في لبنان؟

**Answer:**

يُسمح للعمال السوريين بالعمل قانونياً في لبنان في قطاعات الزراعة والبناء والبيئة. كما يُسمح لهم بالعمل كحمالين أو بوابين بحرية دون تطبيق نظام الحصص المخصصة للمواطنين اللبنانيين.

**Sources:** دليل عن حقوق العمل في لبنان.pdf (Page 38), دليل العمل الحر والمشاريع الصغيرة في لبنان .pdf (Page 19), دليل العمل الحر والمشاريع الصغيرة في لبنان .pdf (Page 47), دليل العمل الحر والمشاريع الصغيرة في لبنان .pdf (Page 12), دليل عن حقوق العمل في لبنان.pdf (Page 39)

---

## Evaluation

In [70]:
# a dict keyed by question string
ground_truth_en = {
    
    # EMPLOYMENT RIGHTS GUIDE

    "What is the legally mandated number of annual leave days?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [16],
        "answerable": True
    },

    "How does annual leave increase based on years of service?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [16],
        "answerable": True
    },

    "What is the official minimum wage?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [13],
        "answerable": True
    },

    "What are the maximum legal working hours per week?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [15],
        "answerable": True
    },

    "How is overtime compensated?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [15],
        "answerable": True
    },

    "What are the rules governing sick leave?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [17],
        "answerable": True
    },

    "Is maternity leave paid and for how long?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [17],
        "answerable": True
    },

    "What rights exist during the probation period?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [12],
        "answerable": True
    },

    "What prior notice is required before termination?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [21, 24],
        "answerable": True
    },

    "What compensation is due in case of abusive dismissal?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [21],
        "answerable": True
    },

    "Under what conditions can employment be terminated due to force majeure?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [22, 23],
        "answerable": True
    },

    "What are the employer's obligations regarding social security contributions?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [14, 15],
        "answerable": True
    },

    "Are foreign workers entitled to social security benefits?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [15, 34],
        "answerable": True
    },

    "Are employees protected from dismissal while on leave?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [16, 17],
        "answerable": True
    },

    "What are the procedures for challenging abusive dismissal?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [24],
        "answerable": True
    },

    "Does the guide regulate cryptocurrency salary payments?": {
        "document": None,
        "pages": [],
        "answerable": False
    },

    "Does the guide contain rules about remote work or work-from-home policies?": {
        "document": None,
        "pages": [],
        "answerable": False
    },


    # SELF-EMPLOYMENT GUIDE

    "How does the guide define workers not linked to an employer?": {
        "document": "Self-Employment-Guide.pdf",
        "pages": [26, 27],
        "answerable": True
    },

    "Are self-employed individuals covered by social security?": {
        "document": "Self-Employment-Guide.pdf",
        "pages": [26],
        "answerable": True
    },

    "Are foreigners eligible for social security benefits?": {
        "document": "Self-Employment-Guide.pdf",
        "pages": [27],
        "answerable": True
    },

    "When is a daily worker considered an employee versus self-employed?": {
        "document": "Self-Employment-Guide.pdf",
        "pages": [26, 27],
        "answerable": True
    },

    "Must individuals conducting commercial activities register as traders?": {
        "document": "Self-Employment-Guide.pdf",
        "pages": [28, 30],
        "answerable": True
    },

    "What activities are considered commercial in nature?": {
        "document": "Self-Employment-Guide.pdf",
        "pages": [28, 29],
        "answerable": True
    },

    "Are workers who work for multiple employers covered by social security?": {
        "document": "Self-Employment-Guide.pdf",
        "pages": [26, 27],
        "answerable": True
    },

    "Are self-employed individuals required to pay tax?": {
        "document": "Self-Employment-Guide.pdf",
        "pages": [32, 33],
        "answerable": True
    },

    "Are self-employed individuals required to pay municipal fees, and on what basis are they calculated?": {
        "document": "Self-Employment-Guide.pdf",
        "pages": [34],
        "answerable": True
    },

    "Are small-scale daily labourers considered traders under commercial law?": {
        "document": "Self-Employment-Guide.pdf",
        "pages": [27, 30],
        "answerable": True
    },

    "Does the guide set minimum pricing standards for freelance services?": {
        "document": None,
        "pages": [],
        "answerable": False
    },

    "Does the guide provide government grants or startup funding schemes for self-employed workers?": {
        "document": None,
        "pages": [],
        "answerable": False
    },

    "What are the grounds under which an employer can dismiss an employee without indemnity or prior notice?": {
        "document": "Employment-Rights-Guide.pdf",
        "pages": [19, 20],
        "answerable": True
    },
}

In [71]:
ground_truth_ar = {

    # دليل حقوق العمل
    "ما هو العدد القانوني المقرر لأيام الإجازة السنوية؟": {
        "document": "دليل عن حقوق العمل في لبنان.pdf",
        "pages": [16],
        "answerable": True
    },
    "كيف تزداد الإجازة السنوية بناءً على سنوات الخدمة؟": {
        "document": "دليل عن حقوق العمل في لبنان.pdf",
        "pages": [16],
        "answerable": True
    },
    "ما هو الحد الأدنى الرسمي للأجور؟": {
        "document": "دليل عن حقوق العمل في لبنان.pdf",
        "pages": [13],
        "answerable": True
    },
    "ما هو الحد الأقصى القانوني لساعات العمل في الأسبوع؟": {
        "document": "دليل عن حقوق العمل في لبنان.pdf",
        "pages": [15],
        "answerable": True
    },
    "كيف يتم تعويض العمل الإضافي؟": {
        "document": "دليل عن حقوق العمل في لبنان.pdf",
        "pages": [15],
        "answerable": True
    },
    "ما هي القواعد المتعلقة بالإجازة المرضية؟": {
        "document": "دليل عن حقوق العمل في لبنان.pdf",
        "pages": [17],
        "answerable": True
    },
    "هل إجازة الأمومة مدفوعة الأجر وما مدتها؟": {
        "document": "دليل عن حقوق العمل في لبنان.pdf",
        "pages": [17],
        "answerable": True
    },
    "ما هي الحقوق خلال فترة الاختبار؟": {
        "document": "دليل عن حقوق العمل في لبنان.pdf",
        "pages": [12],
        "answerable": True
    },
    "ما هي مدة الإنذار المسبق المطلوبة قبل إنهاء العمل؟": {
        "document": "دليل عن حقوق العمل في لبنان.pdf",
        "pages": [21],
        "answerable": True
    },
    "ما هو التعويض المستحق في حالة الفصل التعسفي؟": {
        "document": "دليل عن حقوق العمل في لبنان.pdf",
        "pages": [21],
        "answerable": True
    },
    "في أي ظروف يمكن إنهاء العمل بسبب القوة القاهرة؟": {
        "document": "دليل عن حقوق العمل في لبنان.pdf",
        "pages": [22, 23],
        "answerable": True
    },
    "ما هي التزامات صاحب العمل تجاه اشتراكات الضمان الاجتماعي؟": {
        "document": "دليل عن حقوق العمل في لبنان",
        "pages": [14],
        "answerable": True
    },
    "هل يحق للعمال الأجانب الاستفادة من الضمان الاجتماعي؟": {
        "document": "دليل عن حقوق العمل في لبنان.pdf",
        "pages": [15, 34],
        "answerable": True
    },
    "هل يُحمى الموظفون من الفصل أثناء الإجازة؟": {
        "document": "دليل عن حقوق العمل في لبنان.pdf",
        "pages": [16, 17],
        "answerable": True
    },
    "ما هي إجراءات الطعن في الفصل التعسفي؟": {
        "document": "دليل عن حقوق العمل في لبنان.pdf",
        "pages": [24],
        "answerable": True
    },
    "هل ينظم الدليل دفع الرواتب بالعملات المشفرة؟": {
        "document": None,
        "pages": [],
        "answerable": False
    },
    "هل يتضمن الدليل قواعد تتعلق بالعمل عن بُعد؟": {
        "document": None,
        "pages": [],
        "answerable": False
    },

    # دليل العمل الحر والأعمال الصغيرة
    "كيف يعرّف الدليل العمال غير المرتبطين بصاحب عمل؟": {
        "document": "دليل العمل الحر والمشاريع الصغيرة في لبنان .pdf",
        "pages": [26, 27],
        "answerable": True
    },
    "هل يشمل الضمان الاجتماعي العمال المستقلين؟": {
        "document": "دليل العمل الحر والمشاريع الصغيرة في لبنان .pdf",
        "pages": [26],
        "answerable": True
    },
    "هل يحق للأجانب الاستفادة من الضمان الاجتماعي؟": {
        "document": "دليل العمل الحر والمشاريع الصغيرة في لبنان .pdf",
        "pages": [27],
        "answerable": True
    },
    "متى يُعتبر العامل اليومي موظفاً بدلاً من عامل مستقل؟": {
        "document": "دليل العمل الحر والمشاريع الصغيرة في لبنان .pdf",
        "pages": [26, 27],
        "answerable": True
    },
    "هل يجب على الأفراد الذين يمارسون النشاط التجاري التسجيل كتجار؟": {
        "document": "دليل العمل الحر والمشاريع الصغيرة في لبنان .pdf",
        "pages": [28, 30],
        "answerable": True
    },
    "ما هي الأنشطة التي تُعدّ تجارية بطبيعتها؟": {
        "document": "دليل العمل الحر والمشاريع الصغيرة في لبنان .pdf",
        "pages": [28, 29],
        "answerable": True
    },
    "هل يشمل الضمان الاجتماعي العمال الذين يعملون لدى أكثر من صاحب عمل؟": {
        "document": "دليل العمل الحر والمشاريع الصغيرة في لبنان .pdf",
        "pages": [26, 27],
        "answerable": True
    },
    "هل يخضع العمال المستقلون لضريبة الدخل؟": {
        "document": "دليل العمل الحر والمشاريع الصغيرة في لبنان .pdf",
        "pages": [32, 33],
        "answerable": True
    },
    "هل يخضع العمال المستقلون لرسوم بلدية وما هو أساس احتسابها؟": {
        "document": "دليل العمل الحر والمشاريع الصغيرة في لبنان .pdf",
        "pages": [34],
        "answerable": True
    },
    "هل يُعتبر العمال اليوميون صغار الحجم تجاراً وفق قانون التجارة؟": {
        "document": "دليل العمل الحر والمشاريع الصغيرة في لبنان .pdf",
        "pages": [27, 30],
        "answerable": True
    },
    "هل يحدد الدليل معايير تسعير دنيا للخدمات المستقلة؟": {
        "document": None,
        "pages": [],
        "answerable": False
    },
    "هل يوفر الدليل منحاً حكومية أو برامج تمويل للعمال المستقلين؟": {
        "document": None,
        "pages": [],
        "answerable": False
    },
    "ما هي القطاعات التي يُسمح للعمال السوريين بالعمل فيها قانونياً في لبنان؟": {
        "document": "دليل عن حقوق العمل في لبنان.pdf", 
        "pages": [29, 37, 38],
        "answerable": True
    },
}

In [72]:
ar_chunks = [c for c in chunks if c.metadata.get("language") == "ar"]
sources = set(os.path.basename(c.metadata.get("source", "")) for c in ar_chunks)
print("Arabic chunk sources found:")
for s in sources:
    print(repr(s))

Arabic chunk sources found:
'دليل عن حقوق العمل في لبنان.pdf'
'دليل العمل الحر والمشاريع الصغيرة في لبنان .pdf'


In [73]:
# Unified ground truth: merge both language dicts
# Each key is the question in its original language
ground_truth = {**ground_truth_en, **ground_truth_ar}

print(f"Total ground truth entries: {len(ground_truth)}")

# Check: every evaluation question must have a ground truth entry
missing = [q for q in evaluation_questions_en + evaluation_questions_ar if q not in ground_truth]
if missing:
    print(f"WARNING — missing ground truth for {len(missing)} question(s):")
    for q in missing:
        print(f"  {q}")
else:
    print("All questions have ground truth entries.")

Total ground truth entries: 60
All questions have ground truth entries.


### Behavioral Evaluation

In [75]:
def evaluate_answer_behavior(results, questions, lang_label=""):
    total = len(questions)
    correct_answers = 0
    correct_refusals = 0
    false_refusals = 0      # answerable but system refused
    false_answers = 0       # unanswerable but system tried to answer
    total_answerable = 0
    total_unanswerable = 0

    for r in results:
        q = r["question"]
        gt = ground_truth[q]
        expected_answerable = gt["answerable"]

        is_refusal = (
            "does not provide" in r["answer"].lower()
            or "لا يوفر" in r["answer"]
        )

        if expected_answerable:
            total_answerable += 1
            if not is_refusal:
                correct_answers += 1
            else:
                false_refusals += 1   
        else:
            total_unanswerable += 1
            if is_refusal:
                correct_refusals += 1
            else:
                false_answers += 1

    label = f" ({lang_label})" if lang_label else ""
    print(f"\n=== Answer Behavior Evaluation{label} ===")
    print(f"Total: {total}  |  Answerable: {total_answerable}  |  Unanswerable: {total_unanswerable}")
    print(f"\nCorrect answers:    {correct_answers}/{total_answerable} = {correct_answers/total_answerable:.2f}" if total_answerable else "")
    print(f"False refusals:     {false_refusals}/{total_answerable} (answerable Qs wrongly refused)")
    print(f"Correct refusals:   {correct_refusals}/{total_unanswerable} = {correct_refusals/total_unanswerable:.2f}" if total_unanswerable else "")
    print(f"False answers:      {false_answers}/{total_unanswerable} (unanswerable Qs wrongly answered)")

In [76]:
# Run behavior evaluation
evaluate_answer_behavior(results_en, evaluation_questions_en)
evaluate_answer_behavior(results_ar, evaluation_questions_ar)


=== Answer Behavior Evaluation ===
Total: 30  |  Answerable: 26  |  Unanswerable: 4

Correct answers:    25/26 = 0.96
False refusals:     1/26 (answerable Qs wrongly refused)
Correct refusals:   4/4 = 1.00
False answers:      0/4 (unanswerable Qs wrongly answered)

=== Answer Behavior Evaluation ===
Total: 30  |  Answerable: 26  |  Unanswerable: 4

Correct answers:    23/26 = 0.88
False refusals:     3/26 (answerable Qs wrongly refused)
Correct refusals:   4/4 = 1.00
False answers:      0/4 (unanswerable Qs wrongly answered)


### Retrieval Evaluation

#### Precision@K

In [77]:
def precision_at_k(questions, k=3):
    total_retrieved = 0
    total_relevant = 0

    for question in questions:
        gt = ground_truth[question]
        if not gt["answerable"]:
            continue

        gt_doc = gt["document"]
        gt_pages = set(str(p) for p in gt["pages"])

        retrieved_docs = retrieve(question, k=k)

        retrieved_pairs = set(
            (doc["source"], str(doc["page"])) for doc in retrieved_docs
        )

        total_retrieved += len(retrieved_pairs)

        for doc_name, page in retrieved_pairs:
            if doc_name == gt_doc and page in gt_pages:
                total_relevant += 1

    return total_relevant / total_retrieved if total_retrieved else 0

#### Recall@K

In [78]:
def recall_at_k(questions, k=3):
    total = 0
    hits = 0

    for question in questions:
        gt = ground_truth[question]

        if not gt["answerable"]:
            continue

        total += 1

        gt_doc = gt["document"]
        gt_pages = set(str(p) for p in gt["pages"])

        retrieved_docs = retrieve(question, k=k)

        for doc in retrieved_docs:
            if doc["source"] == gt_doc and str(doc["page"]) in gt_pages:
                hits += 1
                break

    return hits / total if total else 0

#### Hit@K

In [79]:
def hit_at_k(questions, k=3):
    hits = 0
    total = 0

    for question in questions:
        gt = ground_truth[question]

        if not gt["answerable"]:
            continue

        total += 1

        gt_doc = gt["document"]
        gt_pages = set(str(p) for p in gt["pages"])

        retrieved_docs = retrieve(question, k=k)

        for doc in retrieved_docs:
            if doc["source"] == gt_doc and str(doc["page"]) in gt_pages:
                hits += 1
                break

    return hits / total if total else 0

#### MRR (Mean Reciprocal Rank)

In [80]:
def mean_reciprocal_rank(questions, k=10):
    total = 0
    rr_sum = 0.0

    for question in questions:
        gt = ground_truth[question]
        if not gt["answerable"]:
            continue

        total += 1
        gt_doc = gt["document"]
        gt_pages = set(str(p) for p in gt["pages"])
        retrieved_docs = retrieve(question, k=k)

        for rank, doc in enumerate(retrieved_docs, start=1):
            if doc["source"] == gt_doc and str(doc["page"]) in gt_pages:
                rr_sum += 1.0 / rank
                break  # only first hit counts

    return rr_sum / total if total else 0.0

In [81]:
for k in [1, 3, 5]:
    print(f"\n-- K = {k} (English) --")
    print(f"Precision@{k}: {precision_at_k(evaluation_questions_en, k):.2f}")
    print(f"Recall@{k}:    {recall_at_k(evaluation_questions_en, k):.2f}")
    print(f"Hit@{k}:       {hit_at_k(evaluation_questions_en, k):.2f}")

print(f"\n-- MRR (English, K=10) --")
print(f"MRR: {mean_reciprocal_rank(evaluation_questions_en, k=10):.3f}")

for k in [1, 3, 5]:
    print(f"\n-- K = {k} (Arabic) --")
    print(f"Precision@{k}: {precision_at_k(evaluation_questions_ar, k):.2f}")
    print(f"Recall@{k}:    {recall_at_k(evaluation_questions_ar, k):.2f}")
    print(f"Hit@{k}:       {hit_at_k(evaluation_questions_ar, k):.2f}")

print(f"\n-- MRR (Arabic, K=10) --")
print(f"MRR: {mean_reciprocal_rank(evaluation_questions_ar, k=10):.3f}")


-- K = 1 (English) --


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

MRR: 0.416


In [82]:
# Standalone print to ensure Arabic Hit@5 and MRR are visible if output was truncated above
print(hit_at_k(evaluation_questions_ar, k=5))
print(mean_reciprocal_rank(evaluation_questions_ar, k=10))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

0.4156898656898655


## LLM Judge

In [85]:
import random

LLM_JUDGE_SAMPLE_SIZE = 8  

def llm_judge_score(question, answer, context_docs):
    """
    Ask GPT to score the answer 1-3 for correctness given the retrieved context.
    1 = wrong or hallucinated
    2 = partially correct
    3 = correct and grounded
    Returns int score and the judge's explanation.
    """
    context = "\n\n".join(
        f"[Source {i+1}]\n{doc['content']}"
        for i, doc in enumerate(context_docs)
    )

    judge_prompt = f"""You are evaluating a legal QA system.

Question: {question}

Retrieved context:
{context}

System answer: {answer}

Score the answer on this scale:
1 = Incorrect or not grounded in the context
2 = Partially correct, missing important detail
3 = Correct and fully supported by the context

Respond with exactly this format:
Score: <1, 2, or 3>
Reason: <one sentence>"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": judge_prompt}],
        temperature=0,
        max_tokens=100
    )
    text = response.choices[0].message.content.strip()

    try:
        score_line = [l for l in text.splitlines() if l.startswith("Score:")][0]
        score = int(score_line.replace("Score:", "").strip())
    except (IndexError, ValueError):
        score = 0  # parse failure

    return score, text


def run_llm_judge(results, sample_size=LLM_JUDGE_SAMPLE_SIZE, lang_label=""):
    # Only judge answerable questions where the system gave an answer
    judgeables = [
        r for r in results
        if ground_truth[r["question"]]["answerable"]
        and "does not provide" not in r["answer"].lower()
        and "لا يوفر" not in r["answer"]
    ]

    sample = random.sample(judgeables, min(sample_size, len(judgeables)))

    scores = []
    print(f"\n=== LLM Judge Sample ({lang_label}, n={len(sample)}) ===")

    for r in sample:
        # Re-retrieve context for the judge (needed for grounding check)
        candidates = retrieve(r["question"], k=CONFIG["top_k_retrieval"])
        context_docs = rerank(r["question"], candidates, top_n=CONFIG["top_k_final"])

        score, explanation = llm_judge_score(r["question"], r["answer"], context_docs)
        scores.append(score)

        print(f"\nQ: {r['question'][:80]}")
        print(f"A: {r['answer'][:120]}")
        print(explanation)

    if scores:
        avg = sum(scores) / len(scores)
        dist = {1: scores.count(1), 2: scores.count(2), 3: scores.count(3)}
        print(f"\nAverage score: {avg:.2f} / 3")
        print(f"Distribution — 1: {dist[1]}  2: {dist[2]}  3: {dist[3]}")

In [86]:
run_llm_judge(results_en, lang_label="English")
run_llm_judge(results_ar, lang_label="Arabic")


=== LLM Judge Sample (English, n=8) ===


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Q: What prior notice is required before termination?
A: The prior notice period required before termination of an open-ended employment contract in Lebanon, according to Articl
Score: 3  
Reason: The answer accurately reflects the prior notice requirements for termination under Article 50 of the Labour Code, distinguishes between abusive and non-abusive termination, and includes the necessary details about notice delivery and consequences of non-compliance as supported by the provided context.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Q: Are foreigners eligible for social security benefits?
A: Foreigners working in Lebanon are entitled to social security benefits provided they have valid work permits and their c
Score: 3  
Reason: The answer accurately reflects the conditions under which foreigners are eligible for social security benefits in Lebanon, including the requirement of valid work permits and reciprocal agreements, as supported by the provided context.


...


Q: ما هي القطاعات التي يُسمح للعمال السوريين بالعمل فيها قانونياً في لبنان؟
A: يُسمح للعمال السوريين بالعمل قانونياً في لبنان في قطاعات الزراعة والبناء والبيئة. كما يُسمح لهم بالعمل كحمالين أو بوابين
Score: 3  
Reason: الجواب صحيح ومطابق للمعلومات الواردة في المصادر التي تشير إلى السماح للسوريين بالعمل في قطاعات الزراعة والبناء والبيئة دون قيود الحصص المخصصة للبنانيين.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Q: في أي ظروف يمكن إنهاء العمل بسبب القوة القاهرة؟
A: يمكن إنهاء العمل بسبب القوة القاهرة في حالات مثل تقليص حجم المؤسسة، أو استبدال نظام إنتاجي معين بآخر، أو في حالة توقف ال
Score: 3  
Reason: الإجابة صحيحة وشاملة، وتغطي الظروف التي يمكن فيها إنهاء العمل بسبب القوة القاهرة مع الإشارة إلى الإجراءات القانونية والمواقف القضائية المتعلقة بها كما وردت في السياق.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Q: هل يخضع العمال المستقلون لضريبة الدخل؟
A: نعم، يخضع العمال المستقلون لضريبة الدخل على دخلهم الشخصي وفقاً لما ورد في المصدر 4.
Score: 1  
Reason: The provided context does not mention ضريبة الدخل أو العمال المستقلين، ولا يوجد مصدر 4 في المعلومات المسترجعة.

Average score: 2.75 / 3
Distribution — 1: 1  2: 0  3: 7


## Retrieval Evaluation Summary

**Metrics**

- **Precision@K** — of the top K retrieved chunks, what fraction are actually relevant. Measures noise.
- **Recall@K** — does the correct chunk appear anywhere in the top K results. Measures coverage.
- **Hit@K** — binary version of Recall@K: did at least one correct chunk appear in the top K.
- **MRR** — Mean Reciprocal Rank: average of 1/rank of the first correct hit. Measures how high the correct chunk ranks.

**Results**

| K | Precision | Recall / Hit | MRR |
|---|---|---|---|
| 1 | EN: 0.52 / AR: 0.23 | EN: 0.85 / AR: 0.42 | EN: 0.822 / AR: 0.416 |
| 3 | EN: 0.24 / AR: 0.15 | EN: 1.00 / AR: 0.62 | — |
| 5 | EN: 0.17 / AR: 0.12 | EN: 1.00 / AR: 0.81 | — |

**Interpretation**

English retrieval is strong: the correct chunk appears in the top 3 for every answerable question (Recall@3 = 1.00), and MRR of 0.822 means the correct result lands at position 1 or 2 on average. Precision falling with K is expected — broader retrieval introduces noise, which is why reranking exists.

Arabic retrieval is weaker across all metrics. Recall@5 of 0.81 means roughly 1 in 5 Arabic questions have no correct chunk in the top 5 at all, directly causing false refusals downstream. The gap with English is attributed to BM25's limited handling of Arabic morphology and multilingual embedding tradeoffs — both documented limitations of the current approach.

The Precision@1 / Recall@3 tradeoff is a deliberate design choice: the system retrieves broadly for coverage, then relies on the cross-encoder reranker to surface the best chunks for generation.

## Answer Behavior Evaluation

**Setup:** 
30 questions (26 answerable, 4 unanswerable) — English and Arabic. 

Model: `gpt-4.1-mini`

final top-K: 3 (English), 5 (Arabic).

**Results**

| | English | Arabic |
|---|---|---|
| Correct answers | 25/26 (0.96) | 23/26 (0.88) |
| False refusals | 1/26 | 3/26 |
| Correct refusals | 4/4 (1.00) | 4/4 (1.00) |
| False answers | 0/4 | 0/4 |

**Interpretation**

Refusal behavior is fully reliable in both languages — every unanswerable question was correctly rejected with no hallucinated answers. English answer accuracy is strong at 0.96. Arabic improved from 0.77 to 0.88 after increasing the final context window to top-5, recovering 3 previously false-refused questions. The remaining 3 Arabic false refusals trace directly to retrieval failures (Recall@5 = 0.81) rather than generation errors — the LLM correctly refuses when the retrieved context contains no relevant information.

## LLM Judge Evaluation

A sample of 8 answerable questions per language was scored by `gpt-4.1-mini` on a 1–3 scale (1 = incorrect/hallucinated, 2 = partially correct, 3 = correct and grounded).

| | English | Arabic |
|---|---|---|
| Average score | 3.00 / 3 | 2.75 / 3 |
| Score 3 (correct) | 8/8 | 7/8 |
| Score 1 (incorrect) | 0/8 | 1/8 |

English answers were fully grounded in all sampled cases. The single Arabic score-1 case was a judge error — the judge incorrectly claimed the retrieved context contained no relevant information when it did. This highlights a known limitation of LLM-as-judge evaluation: the judge itself is a noisy evaluator and can misread retrieved context, particularly in Arabic.

## Limitations

- Evaluation is conducted on a small manually curated dataset (30 questions). Results should be interpreted as directional rather than statistically definitive.
- Behavioral accuracy (answer vs. refusal) is a proxy metric — it does not measure whether answers are factually correct, only whether the system attempted to answer.
- Arabic retrieval underperforms English due to BM25's limited handling of Arabic morphology and multilingual embedding tradeoffs. Roughly 19% of Arabic questions have no correct chunk in the top 5.
- Hybrid retrieval weights (dense/sparse) are set as fixed heuristics and have not been formally tuned per language.
- LLM-as-judge evaluation is itself imperfect — the judge occasionally misreads retrieved context, as observed in one Arabic scoring case.

## Future Improvements

- Improve Arabic retrieval with a morphology-aware tokenizer (e.g. CAMeL Tools, once dependency conflicts are resolved) and language-specific hybrid weight tuning.
- Introduce semantic answer evaluation using embedding similarity or a stronger judge model.
- Experiment with dynamic top-K retrieval — adjusting K per query based on retrieval confidence rather than a fixed value.
- Extend the ground truth dataset to improve statistical reliability of evaluation results.
- Improve Precision@1 through reranker fine-tuning or query expansion